# Homework 7 - Streaming with Kafka (Redpanda) and PyFlink

Using Green Taxi Trip data from October 2025.

Infrastructure runs via docker compose (Redpanda + Flink + PostgreSQL).

## Q1 - Redpanda version

Run inside the Redpanda container:

```bash
docker exec -it workshop-redpanda-1 rpk version
```

Output: `v24.2.18`

## Setup - download data and install dependencies

In [25]:
!pip install kafka-python pandas pyarrow --quiet


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [46]:
!pip install psycopg2-binary --quiet


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
import urllib.request

url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet'
filename = 'green_tripdata_2025-10.parquet'

print(f'Downloading {filename}...')
urllib.request.urlretrieve(url, filename)
print('Download complete')

Download complete


In [27]:
import pandas as pd

df = pd.read_parquet('green_tripdata_2025-10.parquet')

cols = [
    'lpep_pickup_datetime',
    'lpep_dropoff_datetime',
    'PULocationID',
    'DOLocationID',
    'passenger_count',
    'trip_distance',
    'tip_amount',
    'total_amount'
]

df = df[cols]
print(df.shape)
df.head()

(49416, 8)


,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,tip_amount,total_amount
0,2025-10-01 00:21:47,2025-10-01 00:24:37,247,69,1.0,0.70,1.70,10.00
1,2025-10-01 00:14:03,2025-10-01 00:24:14,66,25,1.0,1.61,2.78,16.68
2,2025-10-01 00:16:44,2025-10-01 00:16:47,244,244,1.0,0.00,2.20,13.20
3,2025-10-01 00:07:36,2025-10-01 00:32:14,95,170,1.0,10.37,11.31,67.85
4,2025-09-30 21:10:29,2025-09-30 21:22:30,82,138,1.0,4.07,6.82,34.12


## Q2 - send data to Redpanda, measure time

In [28]:
import json
from time import time
from kafka import KafkaProducer

producer = KafkaProducer(
    bootstrap_servers='localhost:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

t0 = time()

for _, row in df.iterrows():
    record = row.to_dict()
    # convert datetime to string
    record['lpep_pickup_datetime'] = str(record['lpep_pickup_datetime'])
    record['lpep_dropoff_datetime'] = str(record['lpep_dropoff_datetime'])
    producer.send('green-trips', value=record)

producer.flush()

t1 = time()
print(f'took {(t1 - t0):.2f} seconds')

took 30.80 seconds


## Q3 - consumer, count trips with distance > 5

In [40]:
from kafka import KafkaConsumer

consumer = KafkaConsumer(
    'green-trips',
    bootstrap_servers='localhost:9092',
    auto_offset_reset='earliest',
    value_deserializer=lambda v: json.loads(v.decode('utf-8')),
    consumer_timeout_ms=10000
)

count = 0
total = 0

for msg in consumer:
    total += 1
    if msg.value.get('trip_distance', 0) > 5.0:
        count += 1

print(f'total messages: {total}')
print(f'trips with distance > 5: {count}')

total messages: 148248
trips with distance > 5: 25518


## Q4 - Flink tumbling window (5 min) - pickup location

Job file: `workshop/src/job/q4_tumbling_pickup.py`

Submit with:
```bash
docker exec -it workshop-jobmanager-1 flink run -py /opt/src/job/q4_tumbling_pickup.py
```

Query after results appear in PostgreSQL:
```sql
SELECT PULocationID, num_trips
FROM pickup_counts
ORDER BY num_trips DESC
LIMIT 3;
```

In [41]:
!docker cp q4_tumbling_pickup.py workshop-jobmanager-1:/opt/src/job/q4_tumbling_pickup.py
!docker exec workshop-jobmanager-1 flink run -py /opt/src/job/q4_tumbling_pickup.py

Job has been submitted with JobID f82fb158aaa30adc2e8189f5fe4fbe9d


In [ ]:
q4_job = '''
from pyflink.datastream import StreamExecutionEnvironment
from pyflink.table import StreamTableEnvironment

env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)
t_env = StreamTableEnvironment.create(env)

t_env.execute_sql("""
    CREATE TABLE green_trips (
        lpep_pickup_datetime VARCHAR,
        lpep_dropoff_datetime VARCHAR,
        PULocationID INT,
        DOLocationID INT,
        passenger_count DOUBLE,
        trip_distance DOUBLE,
        tip_amount DOUBLE,
        total_amount DOUBLE,
        event_timestamp AS TO_TIMESTAMP(lpep_pickup_datetime, 'yyyy-MM-dd HH:mm:ss'),
        WATERMARK FOR event_timestamp AS event_timestamp - INTERVAL '5' SECOND
    ) WITH (
        'connector' = 'kafka',
        'topic' = 'green-trips',
        'properties.bootstrap.servers' = 'redpanda:9092',
        'properties.group.id' = 'flink-q4',
        'scan.startup.mode' = 'earliest-offset',
        'format' = 'json'
    )
""")

t_env.execute_sql("""
    CREATE TABLE pickup_counts (
        window_start TIMESTAMP,
        PULocationID INT,
        num_trips BIGINT
    ) WITH (
        'connector' = 'jdbc',
        'url' = 'jdbc:postgresql://postgres:5432/postgres',
        'table-name' = 'pickup_counts',
        'username' = 'postgres',
        'password' = 'postgres'
    )
""")

t_env.execute_sql("""
    INSERT INTO pickup_counts
    SELECT
        TUMBLE_START(event_timestamp, INTERVAL '5' MINUTE) AS window_start,
        PULocationID,
        COUNT(*) AS num_trips
    FROM green_trips
    GROUP BY TUMBLE(event_timestamp, INTERVAL '5' MINUTE), PULocationID
""")
'''

with open('q4_tumbling_pickup.py', 'w') as f:
    f.write(q4_job)

print('job file written')

job file written


In [44]:
!docker cp q4_tumbling_pickup.py workshop-jobmanager-1:/opt/src/job/q4_tumbling_pickup.py
!docker exec workshop-jobmanager-1 flink run -py /opt/src/job/q4_tumbling_pickup.py

Job has been submitted with JobID 73fd1f93362924ac7b8e7435ebaf8086


In [48]:
import psycopg2
import time

# Wait for the job to process data
print("Waiting for Flink job to process data...")
time.sleep(10)

# Connect to PostgreSQL
try:
    conn = psycopg2.connect(
        host="127.0.0.1",
        port=5432,
        database="postgres",
        user="postgres",
        password="postgres",
        connect_timeout=5
    )
    cursor = conn.cursor()

    # Query Q4 results
    cursor.execute("SELECT PULocationID, num_trips FROM pickup_counts ORDER BY num_trips DESC LIMIT 3;")
    print("Q4 - Pickup Counts (Top 3 locations):")
    rows = cursor.fetchall()
    if rows:
        for row in rows:
            print(f"  Location {row[0]}: {row[1]} trips")
    else:
        print("  No data yet")

    conn.close()
except Exception as e:
    print(f"Connection error: {e}")
    print("Make sure PostgreSQL container is running with: docker-compose up -d")

Waiting for Flink job to process data...
Connection error: connection to server at "127.0.0.1", port 5432 failed: FATAL:  password authentication failed for user "postgres"

Make sure PostgreSQL container is running with: docker-compose up -d


## Q5 - Flink session window (5 min gap) - longest streak

Job file: `workshop/src/job/q5_session_window.py`

In [ ]:
q5_job = '''
from pyflink.datastream import StreamExecutionEnvironment
from pyflink.table import StreamTableEnvironment

env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)
t_env = StreamTableEnvironment.create(env)

t_env.execute_sql("""
    CREATE TABLE green_trips (
        lpep_pickup_datetime VARCHAR,
        lpep_dropoff_datetime VARCHAR,
        PULocationID INT,
        DOLocationID INT,
        passenger_count DOUBLE,
        trip_distance DOUBLE,
        tip_amount DOUBLE,
        total_amount DOUBLE,
        event_timestamp AS TO_TIMESTAMP(lpep_pickup_datetime, 'yyyy-MM-dd HH:mm:ss'),
        WATERMARK FOR event_timestamp AS event_timestamp - INTERVAL '5' SECOND
    ) WITH (
        'connector' = 'kafka',
        'topic' = 'green-trips',
        'properties.bootstrap.servers' = 'redpanda:9092',
        'properties.group.id' = 'flink-q5',
        'scan.startup.mode' = 'earliest-offset',
        'format' = 'json'
    )
""")

t_env.execute_sql("""
    CREATE TABLE session_counts (
        window_start TIMESTAMP,
        window_end TIMESTAMP,
        PULocationID INT,
        num_trips BIGINT
    ) WITH (
        'connector' = 'jdbc',
        'url' = 'jdbc:postgresql://postgres:5432/postgres',
        'table-name' = 'session_counts',
        'username' = 'postgres',
        'password' = 'postgres'
    )
""")

t_env.execute_sql("""
    INSERT INTO session_counts
    SELECT
        SESSION_START(event_timestamp, INTERVAL '5' MINUTE) AS window_start,
        SESSION_END(event_timestamp, INTERVAL '5' MINUTE) AS window_end,
        PULocationID,
        COUNT(*) AS num_trips
    FROM green_trips
    GROUP BY SESSION(event_timestamp, INTERVAL '5' MINUTE), PULocationID
""")
'''

with open('q5_session_window.py', 'w') as f:
    f.write(q5_job)

print('job file written')

job file written


In [39]:
!docker cp q5_session_window.py workshop-jobmanager-1:/opt/src/job/q5_session_window.py
!docker exec workshop-jobmanager-1 flink run -py /opt/src/job/q5_session_window.py

Job has been submitted with JobID 18953edcb98c0a35ccbdd4a3eec777f4


In [ ]:
import psycopg2
import time

# Wait for the job to process data
print("Waiting for Flink job to process data...")
time.sleep(10)

# Connect to PostgreSQL
try:
    conn = psycopg2.connect(
        host="127.0.0.1",
        port=5432,
        database="postgres",
        user="postgres",
        password="postgres",
        connect_timeout=5
    )
    cursor = conn.cursor()

    # Query Q5 results
    cursor.execute("SELECT PULocationID, num_trips FROM session_counts ORDER BY num_trips DESC LIMIT 3;")
    print("Q5 - Session Counts (Top 3 locations):")
    rows = cursor.fetchall()
    if rows:
        for row in rows:
            print(f"  Location {row[0]}: {row[1]} trips")
    else:
        print("  No data yet")

    conn.close()
except Exception as e:
    print(f"Connection error: {e}")
    print("Make sure PostgreSQL container is running with: docker-compose up -d")

## Q6 - Flink tumbling window (1 hour) - largest tip amount

Job file: `workshop/src/job/q6_tip_per_hour.py`

In [ ]:
q6_job = '''
from pyflink.datastream import StreamExecutionEnvironment
from pyflink.table import StreamTableEnvironment

env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)
t_env = StreamTableEnvironment.create(env)

t_env.execute_sql("""
    CREATE TABLE green_trips (
        lpep_pickup_datetime VARCHAR,
        lpep_dropoff_datetime VARCHAR,
        PULocationID INT,
        DOLocationID INT,
        passenger_count DOUBLE,
        trip_distance DOUBLE,
        tip_amount DOUBLE,
        total_amount DOUBLE,
        event_timestamp AS TO_TIMESTAMP(lpep_pickup_datetime, 'yyyy-MM-dd HH:mm:ss'),
        WATERMARK FOR event_timestamp AS event_timestamp - INTERVAL '5' SECOND
    ) WITH (
        'connector' = 'kafka',
        'topic' = 'green-trips',
        'properties.bootstrap.servers' = 'redpanda:9092',
        'properties.group.id' = 'flink-q6',
        'scan.startup.mode' = 'earliest-offset',
        'format' = 'json'
    )
""")

t_env.execute_sql("""
    CREATE TABLE tip_per_hour (
        window_start TIMESTAMP,
        total_tip DOUBLE
    ) WITH (
        'connector' = 'jdbc',
        'url' = 'jdbc:postgresql://postgres:5432/postgres',
        'table-name' = 'tip_per_hour',
        'username' = 'postgres',
        'password' = 'postgres'
    )
""")

t_env.execute_sql("""
    INSERT INTO tip_per_hour
    SELECT
        TUMBLE_START(event_timestamp, INTERVAL '1' HOUR) AS window_start,
        SUM(tip_amount) AS total_tip
    FROM green_trips
    GROUP BY TUMBLE(event_timestamp, INTERVAL '1' HOUR)
""")
'''

with open('q6_tip_per_hour.py', 'w') as f:
    f.write(q6_job)

print('job file written')

job file written


In [38]:
!docker cp q6_tip_per_hour.py workshop-jobmanager-1:/opt/src/job/q6_tip_per_hour.py
!docker exec workshop-jobmanager-1 flink run -py /opt/src/job/q6_tip_per_hour.py

Job has been submitted with JobID 31ab344d286660452e70c3b828db4e40


In [ ]:
import psycopg2
import time

# Wait for the job to process data
print("Waiting for Flink job to process data...")
time.sleep(10)

# Connect to PostgreSQL
try:
    conn = psycopg2.connect(
        host="127.0.0.1",
        port=5432,
        database="postgres",
        user="postgres",
        password="postgres",
        connect_timeout=5
    )
    cursor = conn.cursor()

    # Query Q6 results
    cursor.execute("SELECT window_start, total_tip FROM tip_per_hour ORDER BY total_tip DESC LIMIT 3;")
    print("Q6 - Tip Per Hour (Top 3 hours):")
    rows = cursor.fetchall()
    if rows:
        for row in rows:
            print(f"  {row[0]}: ${row[1]:.2f}")
    else:
        print("  No data yet")

    conn.close()
except Exception as e:
    print(f"Connection error: {e}")
    print("Make sure PostgreSQL container is running with: docker-compose up -d")